In [1]:
# Env
NUM_AGENTS = 2
HEIGHT = 6
WIDTH = 6
SPAWN_PROB_PER_CELL = 0.05
DESPAWN_PROB_PER_CELL = 0.09

# Training
DISCOUNT = 0.99
EPSILON = 0.1
LEARNING_RATE = 0.00002
BATCH_SIZE = 32
TARGET_UPDATE_FREQUENCY = 100

# Model
CONV_CHANNELS = [16, 32]
HIDDEN_FEATURES = 128
HIDDEN_LAYERS = 2
KERNEL_SIZE = 3


In [2]:
import sys
sys.path.append('../../../')
from models.value_cnn import ValueCNNDecentralized, ValueCNNCentralized
import torch
from tadd_helpers.env_functions import State
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

--- PyTorch is configured to use: cpu ---


In [3]:
decentralized_model_2a_path = "decentralized_model6x6_agents2/decentralized_value_cnn_agent_i.pt"
decentralized_model_2a_paths = []
for i in range(2):
    decentralized_model_2a_paths.append(decentralized_model_2a_path.replace(f"agent_i", f"agent_{i}"))
print(decentralized_model_2a_paths)

centralized_model_2a_path = "centralized_model6x6_agents2/centralized_value_cnn.pt"

['decentralized_model6x6_agents2/decentralized_value_cnn_agent_0.pt', 'decentralized_model6x6_agents2/decentralized_value_cnn_agent_1.pt']


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
decentralized_2a_cnns: list[ValueCNNDecentralized] = []
for i in range(len(decentralized_model_2a_paths)):
    model_path = decentralized_model_2a_paths[i]
    cnn = ValueCNNDecentralized(HEIGHT, WIDTH, LEARNING_RATE, DISCOUNT, HIDDEN_FEATURES, HIDDEN_LAYERS, CONV_CHANNELS, KERNEL_SIZE)
    
    state_dict = torch.load(model_path, map_location=device)
    cnn.load_state_dict(state_dict)
    # --------------------
    
    # 3. Explicitly move the entire model to the chosen device
    cnn.to(device)

    cnn.eval()
    decentralized_2a_cnns.append(cnn)
centralized_2a_cnn = ValueCNNCentralized(HEIGHT, WIDTH, LEARNING_RATE, DISCOUNT, HIDDEN_FEATURES, HIDDEN_LAYERS, CONV_CHANNELS, KERNEL_SIZE)
centralized_2a_cnn.load_state_dict(torch.load(centralized_model_2a_path, map_location=device))
centralized_2a_cnn.to(device)
centralized_2a_cnn.eval()


ValueCNNCentralized(
  (conv_layers): ModuleList(
    (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (mlp_head): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
  (target_net): CNNCentralized(
    (conv_layers): ModuleList(
      (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): Max

In [5]:
states: list[State] = []
states_file_path = "centralized_model6x6_agents2/trained_states_centralized.pkl"
with open(states_file_path, "rb") as f:
    states = pickle.load(f)
print(states[:3])

[State(_apples=array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]]), _agents={0: (3, 4), 1: (2, 4)}, name='Empty State'), State(_apples=array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]]), _agents={0: (3, 4), 1: (np.int64(1), np.int64(4))}, name='Empty State'), State(_apples=array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0]]), _agents={0: (3, 4), 1: (np.int64(1), np.int64(4))}, name='Empty State')]


In [ ]:
print(len(states))
states_to_eval = states

800000


In [7]:
all_results = []
for t, state in tqdm(enumerate(states_to_eval)):
    val_d = []
    for i in range(NUM_AGENTS):
        cnn = decentralized_2a_cnns[i]
        raw_dict = cnn.state_to_raw_dict(state)
        pred_value = cnn.get_model_reward_prediction_from_raw(raw_dict, agent_pos=state.agent_position(i))
        val_d.append(pred_value.item())
        
    valC = centralized_2a_cnn.get_model_reward_prediction_from_raw(centralized_2a_cnn.state_to_raw_dict(state)).item()
    
    val_d_team = sum(val_d)
    result_info = {
        "timestep": (t / 2),
        "centralized_team_value": valC,
        "decentralized_team_value": val_d_team,
        "error": abs(valC - val_d_team),
        "centralized - decentralized": valC - val_d_team,
        "state_object": state
        
    }
    all_results.append(result_info)

10it [00:00, 298.46it/s]


In [10]:
df = pd.DataFrame(all_results)
print(df.head())

   timestep  centralized_team_value  decentralized_team_value     error  \
0       0.0               25.828159                 26.031549  0.203389   
1       0.5               25.774891                 26.388330  0.613440   
2       1.0               27.340803                 27.258855  0.081948   
3       1.5               27.311968                 27.621363  0.309395   
4       2.0               26.131105                 27.006741  0.875635   

   centralized - decentralized  \
0                    -0.203389   
1                    -0.613440   
2                     0.081948   
3                    -0.309395   
4                    -0.875635   

                                        state_object  
0  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
1  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
2  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
3  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  
4  --- Empty State (Grid: 6x6) ---\n\n--- Agent L...  


In [9]:
with open("predicted_trajectory_values_6x6_agents2.pkl", "wb") as f:
    pickle.dump(df, f)